# Generate test-set probabilities from the adaptive-weighting checkpoints (fixes bootstrap CI)


No retraining, no GPU needed (104 images, small CNN). This notebook exists because the adaptive-weighting fine-tuning run (`ssl-finetune-adaptive-weights.ipynb`) only saved model checkpoints (`adaptive_best_seed_*.pth`) and an aggregate results JSON -- it never wrote out per-image test-set probabilities, which is what `bootstrap-ci.ipynb` needs for the adaptive-weight bootstrap CI. This loads each seed's checkpoint, runs one inference pass over the unchanged `expA_test.csv`, and writes `test_probs_seed{seed}.csv` in the exact same format as the fixed-weight `test_probs_seed{seed}.csv` files, so `bootstrap-ci.ipynb`'s `ADAPTIVE_PATTERN` can point straight at this notebook's output. Edit the three `EDIT THESE` paths below.

In [1]:
# ============================================================
# Generate per-image test-set softmax probabilities from the
# adaptive-weighting checkpoints, in the same CSV format as the
# fixed-weight test_probs_seed{seed}.csv files.
#
# WHY THIS NOTEBOOK EXISTS:
#   ssl-finetune-adaptive-weights.ipynb (the notebook that trained
#   the adaptive-weight model) only saved:
#     - adaptive_best_seed_{42,123,2024}.pth  (model checkpoints)
#     - ssl_adaptive_weight_3seed_results.json (aggregate metrics)
#   It never wrote out per-image test-set probabilities. That is
#   exactly what bootstrap-ci.ipynb needs for a bootstrap CI (it
#   resamples individual test predictions, not aggregate metrics).
#   Pointing ADAPTIVE_PATTERN at val_probs_seed{seed}.csv (the
#   7-MEL validation split) was the earlier bug -- there was no
#   correct file to point at until this notebook produces one.
#
# DATASETS NEEDED (edit the paths below):
#   - expA_test.csv + LABELED_DIR (ISIC images) -- same 104-image
#     test set used everywhere else in the paper. UNCHANGED, not
#     re-split, not oversampled -- one plain inference pass.
#   - ADAPTIVE_CKPT_DIR: the folder containing the three
#     adaptive_best_seed_{42,123,2024}.pth checkpoints (this is
#     ssl-finetune-adaptive-weights-pth.zip, unzipped).
#
# ARCHITECTURE:
#   SimpleEncoder + SSLClassifier, identical class definitions to
#   ssl-finetune-adaptive-weights.ipynb, so the saved state_dict
#   (encoder.* + classifier.* keys) loads directly -- no prefix
#   stripping needed, but load is still tolerant (strict=False)
#   with a warning if anything doesn't line up.
#
# OUTPUT:
#   - test_probs_seed{42,123,2024}.csv, columns: BKL, MEL, NV,
#     true_label -- one row per test image, same schema as the
#     fixed-weight files. Point bootstrap-ci.ipynb's
#     ADAPTIVE_PATTERN at wherever you upload these as a Kaggle
#     dataset, e.g.
#     '.../adaptive-test-probs/test_probs_seed{seed}.csv'
# ============================================================
import os, glob
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ---- EDIT THESE ----
BACKUP_DIR       = '/kaggle/input/datasets/ittisamurtunib/dataset/CSVs-20260711T054424Z-2-001/CSVs'
LABELED_DIR      = '/kaggle/input/datasets/ittisamurtunib/dataset/ISIC_2019_Training_Input/ISIC_2019_Training_Input'
ADAPTIVE_CKPT_DIR = '/kaggle/input/datasets/ittisamurtunib/ssl-finetune-adaptive-weights-pth'
# ---------------------

OUTPUT_DIR = '/kaggle/working'
SEEDS = [42, 123, 2024]
CLASSES = ['BKL', 'MEL', 'NV']  # must match sorted(train_df['label'].unique()) order used at training time

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)), transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])


def resolve_path(img_name):
    if not img_name.endswith('.jpg'):
        img_name += '.jpg'
    p = os.path.join(LABELED_DIR, img_name)
    if not os.path.exists(p):
        m = glob.glob(f"{LABELED_DIR}/**/{img_name}", recursive=True)
        p = m[0] if m else p
    return p


class TestDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(resolve_path(str(row['image']))).convert('RGB')
        return eval_transform(img), row['label']


class SimpleEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
            nn.AdaptiveAvgPool2d(1), nn.Flatten()
        )

    def forward(self, x):
        return self.features(x)


class SSLClassifier(nn.Module):
    def __init__(self, encoder, num_classes):
        super().__init__()
        self.encoder = encoder
        self.classifier = nn.Sequential(nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.3), nn.Linear(64, num_classes))

    def forward(self, x):
        return self.classifier(self.encoder(x))


def load_full_ckpt(model, ckpt_path):
    state = torch.load(ckpt_path, map_location=device)
    missing, unexpected = model.load_state_dict(state, strict=False)
    if missing or unexpected:
        print(f"  [warning] checkpoint load: {len(missing)} missing, {len(unexpected)} unexpected keys "
              f"-- verify {ckpt_path} is the right file.")
    return model


test_df = pd.read_csv(os.path.join(BACKUP_DIR, 'expA_test.csv'))
n_mel = int((test_df['label'] == 'MEL').sum())
assert n_mel == 8, (
    f"expA_test.csv has {n_mel} MEL rows, expected 8 -- make sure BACKUP_DIR points at the "
    f"same CSVs used for the fixed-weight test_probs_seed*.csv files, not a different split."
)
test_loader = DataLoader(TestDataset(test_df), batch_size=16, shuffle=False)

for seed in SEEDS:
    ckpt_path = os.path.join(ADAPTIVE_CKPT_DIR, f'adaptive_best_seed_{seed}.pth')
    model = SSLClassifier(SimpleEncoder(), len(CLASSES)).to(device)
    model = load_full_ckpt(model, ckpt_path)
    model.eval()

    all_probs, all_true = [], []
    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            probs = F.softmax(model(images), dim=1).cpu().numpy()
            all_probs.append(probs)
            all_true.extend(labels)

    probs_arr = np.concatenate(all_probs, axis=0)
    out_df = pd.DataFrame(probs_arr, columns=CLASSES)
    out_df['true_label'] = all_true

    out_path = f'{OUTPUT_DIR}/test_probs_seed{seed}.csv'
    out_df.to_csv(out_path, index=False)
    seed_mel_recall = float(
        ((out_df[CLASSES].values.argmax(axis=1) == CLASSES.index('MEL')) & (out_df['true_label'] == 'MEL')).sum()
        / max(1, (out_df['true_label'] == 'MEL').sum())
    )
    print(f"[seed {seed}] wrote {out_path} ({len(out_df)} rows, {int((out_df['true_label']=='MEL').sum())} MEL) "
          f"-- MEL recall check: {seed_mel_recall:.3f}")

print("\nDone. Upload test_probs_seed{42,123,2024}.csv as a Kaggle dataset, then point")
print("bootstrap-ci.ipynb's ADAPTIVE_PATTERN at '.../<your-dataset-name>/test_probs_seed{seed}.csv'.")


[seed 42] wrote /kaggle/working/test_probs_seed42.csv (104 rows, 8 MEL) -- MEL recall check: 0.500
[seed 123] wrote /kaggle/working/test_probs_seed123.csv (104 rows, 8 MEL) -- MEL recall check: 0.500
[seed 2024] wrote /kaggle/working/test_probs_seed2024.csv (104 rows, 8 MEL) -- MEL recall check: 0.375

Done. Upload test_probs_seed{42,123,2024}.csv as a Kaggle dataset, then point
bootstrap-ci.ipynb's ADAPTIVE_PATTERN at '.../<your-dataset-name>/test_probs_seed{seed}.csv'.
